# Reinforcement Learning Lab: REINFORCE on MountainCar-v0


## Scenario: MountainCar

In MountainCar, a car is stuck between two hills. The engine is weak, so the car cannot directly climb the right hill. It must first move left, gain momentum, and then move right to reach the flag.

The agent has 3 actions:

| Action | Meaning |
|---|---|
| 0 | Push left |
| 1 | Do nothing |
| 2 | Push right |

The observation/state has 2 values:

| State value | Meaning |
|---|---|
| position | where the car is |
| velocity | how fast the car is moving |



##What is REINFORCE?

REINFORCE is a **policy-gradient algorithm**.

In simple words:

> REINFORCE directly learns a policy that tells the agent which action to take.

A **policy** gives probabilities for actions.

Example:

| Action | Probability |
|---|---:|
| Push left | 0.20 |
| Do nothing | 0.10 |
| Push right | 0.70 |

The agent samples an action using these probabilities.

If an action leads to good future rewards, REINFORCE increases the probability of that action.
If an action leads to poor future rewards, REINFORCE decreases the probability of that action.

## Step 1: Install required libraries

Run this cell only if Gymnasium or PyTorch is not already installed.

In [1]:
# Uncomment and run if needed
!pip install gymnasium torch matplotlib numpy

## Step 2: Import libraries

We need:

- `gymnasium` to create the MountainCar environment,
- `torch` to build and train the neural network,
- `numpy` for calculations,
- `matplotlib` for plots.

In [2]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

import random
from collections import deque

## Step 3: Set random seeds

A random seed helps us get more repeatable results.

The result may still vary slightly on different computers, but the training will be more stable.

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Step 4: Create the MountainCar environment

MountainCar has:

- **2 state values**: position and velocity,
- **3 actions**:
  - `0`: push left,
  - `1`: no push,
  - `2`: push right.

The goal is to move the car to the flag on the right side.

In [4]:
env = gym.make("MountainCar-v0")
state, info = env.reset(seed=SEED)

print("Initial state:", state)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)

env.close()

Initial state: [-0.4452088  0.       ]
Observation space: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
Action space: Discrete(3)
Number of actions: 3


## Step 5: Understand success and failure

In MountainCar-v0:

- success means the car position becomes `>= 0.5`,
- failure usually means the episode reaches 200 steps without reaching the flag.

If the total original reward is `-200`, that usually means failure.

If the reward is greater than `-200`, it means the episode ended earlier, usually because the car reached the flag.

In [5]:
def is_success(state):
    """Return True if the car has reached the flag."""
    position = state[0]
    return position >= 0.5

## Step 6: Policy network

REINFORCE learns a **policy**.

A policy means:

> given the current state, choose an action.

The neural network receives the state `(position, velocity)` and outputs action scores for the 3 actions.

Then we use `softmax` to convert those scores into probabilities.

In [6]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, state):
        logits = self.model(state)
        return logits

    def get_action(self, state, deterministic=False):
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        logits = self.forward(state_tensor)
        probs = torch.softmax(logits, dim=-1)

        if deterministic:
            action = torch.argmax(probs, dim=-1)
            log_prob = torch.log(probs[0, action] + 1e-8)
            entropy = Categorical(probs).entropy()
        else:
            distribution = Categorical(probs)
            action = distribution.sample()
            log_prob = distribution.log_prob(action)
            entropy = distribution.entropy()

        return action.item(), log_prob.squeeze(), entropy.squeeze()

## Step 7: Reward shaping

The original MountainCar reward is `-1` at every step.

This is hard for REINFORCE because the agent does not know which action helped.

So we add extra training signals:

- reward for moving higher on the right side,
- reward for having speed,
- big bonus when the car reaches the flag.

This does **not** change the real goal. It only helps training.

In [7]:
def shaped_reward_function(state, next_state, original_reward, terminated):
    """
    This reward is used only for training.
    It gives the agent extra hints about useful behavior.
    """
    position, velocity = next_state

    # Encourage the car to move fast because momentum is important.
    speed_bonus = 10.0 * abs(velocity)

    # Encourage the car to move toward the right hill.
    # position ranges roughly from -1.2 to 0.6.
    position_bonus = 5.0 * (position + 0.5)

    # Give a strong bonus if the flag is reached.
    goal_bonus = 100.0 if position >= 0.5 else 0.0

    shaped_reward = original_reward + speed_bonus + position_bonus + goal_bonus
    return shaped_reward

## Step 8: Discounted returns

REINFORCE uses the total future reward after each action.

This is called the **return**.

We also normalize returns to make training more stable.

In [8]:
def compute_discounted_returns(rewards, gamma=0.99):
    returns = []
    G = 0
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)

    returns = torch.tensor(returns, dtype=torch.float32)

    # Normalize returns for stable learning
    if len(returns) > 1:
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    return returns

## Step 9: Train REINFORCE until success

This training function is updated to avoid the problem where all episodes fail.

It trains for many episodes and saves the best model.

The function prints:

- average original reward,
- number of successful episodes in the last 50 episodes.

The training stops early when enough successes are found.

In [9]:
def train_reinforce_until_success(
    max_episodes=5000,
    gamma=0.99,
    learning_rate=0.0007,
    entropy_beta=0.01,
    target_successes_in_last_50=5,
    print_every=50,
    save_path="best_reinforce_mountaincar.pth"
):
    env = gym.make("MountainCar-v0")
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy = PolicyNetwork(state_dim, action_dim)
    optimizer = optim.Adam(policy.parameters(), lr=learning_rate)

    original_rewards_history = []
    shaped_rewards_history = []
    success_history = []

    best_original_reward = -9999
    best_success_count = 0

    for episode in range(1, max_episodes + 1):
        state, info = env.reset(seed=SEED + episode)

        log_probs = []
        entropies = []
        shaped_rewards = []
        original_rewards = []

        terminated = False
        truncated = False

        while not (terminated or truncated):
            action, log_prob, entropy = policy.get_action(state, deterministic=False)
            next_state, original_reward, terminated, truncated, info = env.step(action)

            shaped_reward = shaped_reward_function(state, next_state, original_reward, terminated)

            log_probs.append(log_prob)
            entropies.append(entropy)
            shaped_rewards.append(shaped_reward)
            original_rewards.append(original_reward)

            state = next_state

        total_original_reward = sum(original_rewards)
        total_shaped_reward = sum(shaped_rewards)
        success = 1 if state[0] >= 0.5 else 0

        original_rewards_history.append(total_original_reward)
        shaped_rewards_history.append(total_shaped_reward)
        success_history.append(success)

        # Compute REINFORCE loss
        returns = compute_discounted_returns(shaped_rewards, gamma=gamma)
        log_probs_tensor = torch.stack(log_probs)
        entropies_tensor = torch.stack(entropies)

        # Main REINFORCE loss: good actions should become more likely
        policy_loss = -(log_probs_tensor * returns).sum()

        # Entropy loss: keeps exploration alive
        entropy_loss = -entropy_beta * entropies_tensor.sum()

        loss = policy_loss + entropy_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=1.0)
        optimizer.step()

        # Save best model
        recent_successes = sum(success_history[-50:])
        if total_original_reward > best_original_reward or recent_successes > best_success_count:
            best_original_reward = total_original_reward
            best_success_count = recent_successes
            torch.save(policy.state_dict(), save_path)

        if episode % print_every == 0:
            avg_original = np.mean(original_rewards_history[-50:])
            avg_shaped = np.mean(shaped_rewards_history[-50:])
            recent_successes = sum(success_history[-50:])
            print(
                f"Episode {episode:4d} | "
                f"Avg original reward: {avg_original:7.2f} | "
                f"Avg shaped reward: {avg_shaped:7.2f} | "
                f"Successes in last 50: {recent_successes}"
            )

        # Early stop when success is observed several times
        if episode >= 200 and sum(success_history[-50:]) >= target_successes_in_last_50:
            print("Training stopped early because the agent reached the success target.")
            print(f"Successes in last 50 episodes: {sum(success_history[-50:])}")
            break

    env.close()

    # Load best model before returning
    policy.load_state_dict(torch.load(save_path, map_location="cpu"))
    print("Training completed. Best model loaded.")

    return policy, original_rewards_history, shaped_rewards_history, success_history

## Step 10: Start training

This may take a few minutes.

If you still see no success, increase `max_episodes` to `8000` or `10000`.

For teaching, first run with `5000` episodes. The function will stop early if it sees enough successful episodes.

In [ ]:
reinforce_policy, original_rewards, shaped_rewards, success_history = train_reinforce_until_success(
    max_episodes=5000,
    target_successes_in_last_50=5,
    print_every=50
)

Episode   50 | Avg original reward: -200.00 | Avg shaped reward: -212.32 | Successes in last 50: 0
Episode  100 | Avg original reward: -200.00 | Avg shaped reward: -201.66 | Successes in last 50: 0
Episode  150 | Avg original reward: -200.00 | Avg shaped reward: -197.35 | Successes in last 50: 0
Episode  200 | Avg original reward: -200.00 | Avg shaped reward: -178.09 | Successes in last 50: 0
Episode  250 | Avg original reward: -200.00 | Avg shaped reward: -147.48 | Successes in last 50: 0
Episode  300 | Avg original reward: -200.00 | Avg shaped reward: -119.47 | Successes in last 50: 0
Episode  350 | Avg original reward: -200.00 | Avg shaped reward: -123.12 | Successes in last 50: 0
Episode  400 | Avg original reward: -200.00 | Avg shaped reward: -124.88 | Successes in last 50: 0
Episode  450 | Avg original reward: -200.00 | Avg shaped reward: -104.21 | Successes in last 50: 0
Episode  500 | Avg original reward: -200.00 | Avg shaped reward:  -82.08 | Successes in last 50: 0
Episode  5

## Step 13: Test the trained model

Now we test the model without exploration.

During testing, the agent chooses the action with the highest probability.

This is called a **deterministic policy**.

In [10]:
def test_model(policy, test_episodes=20, render=False):
    if render:
        env = gym.make("MountainCar-v0", render_mode="human")
    else:
        env = gym.make("MountainCar-v0")

    test_rewards = []
    test_successes = 0
    steps_list = []

    for ep in range(1, test_episodes + 1):
        state, info = env.reset(seed=1000 + ep)
        terminated = False
        truncated = False
        total_reward = 0
        steps = 0

        while not (terminated or truncated):
            action, _, _ = policy.get_action(state, deterministic=True)
            state, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            steps += 1

        success = state[0] >= 0.5
        if success:
            test_successes += 1

        test_rewards.append(total_reward)
        steps_list.append(steps)

        print(
            f"Test Episode {ep:2d} | "
            f"Reward: {total_reward:6.1f} | "
            f"Steps: {steps:3d} | "
            f"Success: {success}"
        )

    env.close()

    print("Testing summary")
    print("Successes:", test_successes, "/", test_episodes)
    print("Average reward:", np.mean(test_rewards))
    print("Average steps:", np.mean(steps_list))

    return test_rewards, steps_list, test_successes

# Test without rendering
test_rewards, test_steps, test_successes = test_model(reinforce_policy, test_episodes=20, render=False)

NameError: name 'reinforce_policy' is not defined